Spatial Transformer Testset Evaluation

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader
import threading
import queue
from numcodecs import Blosc
import shutil


In [ ]:

class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__();self.enc1=self._conv_block(in_channels,16);self.enc2=self._conv_block(16,32);self.pool=nn.MaxPool3d(2);self.bottleneck=self._conv_block(32,64);self.upconv2=nn.ConvTranspose3d(64,32,kernel_size=2,stride=2);self.dec2=self._conv_block(64,32);self.upconv1=nn.ConvTranspose3d(32,16,kernel_size=2,stride=2);self.dec1=self._conv_block(32,16);self.final_conv=nn.Conv3d(16,out_channels,kernel_size=1);self.final_conv.weight.data.zero_();self.final_conv.bias.data.zero_()
    def _conv_block(self,c,o):return nn.Sequential(nn.Conv3d(c,o,3,1,1),nn.ReLU(True),nn.Conv3d(o,o,3,1,1),nn.ReLU(True))
    def forward(self,f,m):x=torch.cat([f,m],dim=1);e1=self.enc1(x);e2=self.enc2(self.pool(e1));b=self.bottleneck(self.pool(e2));d2=self.upconv2(b);d2=torch.cat([d2,e2],dim=1);d2=self.dec2(d2);d1=self.upconv1(d2);d1=torch.cat([d1,e1],dim=1);d1=self.dec1(d1);return self.final_conv(d1)

class SpatialTransformer3D(nn.Module):
    def __init__(self,size):
        super().__init__();v=[torch.arange(0,s)for s in size];g=torch.meshgrid(v,indexing='ij');g=torch.stack(g).unsqueeze(0);self.register_buffer('grid',g.float(),persistent=False)
    def forward(self,s,f):
        n=self.grid+f;sh=f.shape[2:];
        for i in range(len(sh)):n[:,i,...]=2*(n[:,i,...]/(sh[i]-1)-0.5)
        n=n.permute(0,2,3,4,1);n=n[...,[2,1,0]];return F.grid_sample(s,n,align_corners=True,padding_mode="border")

class InferenceDataset(Dataset):
    def __init__(self, moving_zarr_path, fixed_zarr_path, fixed_index=0):
        super().__init__();self.moving_arr=zarr.open(moving_zarr_path,mode='r');self.fixed_arr=zarr.open(fixed_zarr_path,mode='r')
        self.num_images=self.moving_arr.shape[3];self.original_shape=self.moving_arr.shape;self.padded_shape=[s for s in self.original_shape[:3]]
        for i in range(3):
            if self.padded_shape[i]%4!=0:self.padded_shape[i]=(self.padded_shape[i]//4+1)*4
        self.fixed_np=self.fixed_arr[...,fixed_index]
    def __len__(self):return self.num_images
    def __getitem__(self,idx):
        m_np=self.moving_arr[...,idx];m_t=self._preprocess(m_np);f_t=self._preprocess(self.fixed_np)
        return m_t,f_t
    def _preprocess(self,v_np):
        t=torch.from_numpy(v_np.astype(np.float32)).permute(2,0,1).unsqueeze(0)
        pd=self.padded_shape[2]-t.shape[1];ph=self.padded_shape[0]-t.shape[2];pw=self.padded_shape[1]-t.shape[3]
        p=(pw//2,pw-pw//2,ph//2,ph-ph//2,pd//2,pd-pd//2);return F.pad(t,p,"constant",0)


def save_to_zarr_worker(q, warped_zarr, dvf_zarr):
    while True:
        item = q.get()
        if item is None:
            break
        try:
            i, warped_np, dvf_np = item
            # warped_np der Form (H, W, D) in warped_zarr (H, W, D, T)
            warped_zarr[:, :, :, i] = warped_np
            # dvf_np der Form (H, W, D, 3) in dvf_zarr (H, W, D, T, 3)
            dvf_zarr[:, :, :, i, :] = dvf_np
        except Exception as e:
            print(f"Fehler im Writer-Thread bei Index {i}: {e}")
        finally:
            q.task_done()


if __name__ == '__main__':
    # --- Konfiguration ---
    MODEL_PATH = "supervised_model_1.pth"
    MOVING_TEST_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/DCE"
    FIXED_TEST_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
    OUTPUT_WARPED_PATH = "/media/shooty/Data/MRI_Data/SpatialTransformer_3/warped_images.zarr"
    OUTPUT_DVF_PATH = "/media/shooty/Data/MRI_Data/SpatialTransformer_3/predicted_dvfs.zarr"
    FIXED_IMAGE_INDEX = 1
    NUM_WRITER_THREADS = 8

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")
    
    for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
        if os.path.exists(path):
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
            print(f"Warnung: Bestehende Datei/Verzeichnis {path} wurde gelöscht.")

    print("Lade trainiertes Modell...")
    dataset_for_shape = InferenceDataset(MOVING_TEST_PATH, FIXED_TEST_PATH)
    padded_input_size = (dataset_for_shape.padded_shape[2], dataset_for_shape.padded_shape[0], dataset_for_shape.padded_shape[1])
    model = UNet3D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    try:
        model = torch.compile(model)
        print("Modell erfolgreich mit torch.compile() optimiert.")
    except Exception: print("torch.compile() nicht verfügbar.")
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    
    dataset = InferenceDataset(MOVING_TEST_PATH, FIXED_TEST_PATH, fixed_index=FIXED_IMAGE_INDEX)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=max(1, os.cpu_count() // 2), pin_memory=True, prefetch_factor=2)

    print("Erstelle thread-sichere Ausgabedateien...")
    synchronizer = zarr.ThreadSynchronizer()
    compressor = None
    

    warped_output_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w', shape=dataset.original_shape, chunks=dataset.moving_arr.chunks, dtype=dataset.moving_arr.dtype, compressor=compressor, synchronizer=synchronizer)

    # Korrekte Form: (H, W, D, T, 3)
    dvf_shape = dataset.original_shape + (3,)
    dvf_chunks = dataset.moving_arr.chunks + (3,)

    dvf_output_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w', shape=dvf_shape, chunks=dvf_chunks, dtype='float32', compressor=compressor, synchronizer=synchronizer)
    print(f"Korrekte DVF-Array-Form erstellt: {dvf_output_zarr.shape}")

    data_queue = queue.Queue(maxsize=NUM_WRITER_THREADS * 8)
    writer_threads = []
    print(f"Starte {NUM_WRITER_THREADS} asynchrone Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS):
        thread = threading.Thread(target=save_to_zarr_worker, args=(data_queue, warped_output_zarr, dvf_output_zarr))
        thread.daemon = True
        thread.start(); writer_threads.append(thread)

    print(f"Starte optimierte Inferenz für {len(dataset)} Volumen...")
    original_shape_dims = dataset.original_shape
    padded_shape_dims = dataset.padded_shape
    model.eval()
    with torch.no_grad():
        for i, (moving_batch, fixed_batch) in enumerate(tqdm.tqdm(dataloader, desc="Verarbeite Testdatensatz")):
            moving_batch = moving_batch.to(device, non_blocking=True); fixed_batch = fixed_batch.to(device, non_blocking=True)
            with torch.autocast(device_type=str(device), dtype=torch.float16):
                predicted_dvf_batch = model(fixed_batch, moving_batch); warped_batch = stn(moving_batch, predicted_dvf_batch)
            
            warped_tensor = warped_batch.squeeze(0).cpu(); disp_field = predicted_dvf_batch.squeeze(0).cpu()
            
            pad_d_start = (padded_shape_dims[2] - original_shape_dims[2]) // 2
            pad_h_start = (padded_shape_dims[0] - original_shape_dims[0]) // 2
            pad_w_start = (padded_shape_dims[1] - original_shape_dims[1]) // 2
            
            cropped_warped = warped_tensor[:, pad_d_start:pad_d_start+original_shape_dims[2], pad_h_start:pad_h_start+original_shape_dims[0], pad_w_start:pad_w_start+original_shape_dims[1]]
            cropped_disp = disp_field[:, pad_d_start:pad_d_start+original_shape_dims[2], pad_h_start:pad_h_start+original_shape_dims[0], pad_w_start:pad_w_start+original_shape_dims[1]]
            
            # Form für warped_np: (H, W, D)
            warped_np = cropped_warped.squeeze(0).numpy().transpose(1, 2, 0)
            # Form für dvf_np: (H, W, D, 3)
            dvf_np = cropped_disp.numpy().transpose(2, 3, 1, 0)
            
            data_queue.put((i, warped_np, dvf_np))
            
    print("Hauptprozess abgeschlossen. Sende Stopp-Signal an Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS): data_queue.put(None)
    
    print("Warte, bis alle Writer-Threads ihre Arbeit beendet haben...")
    for thread in writer_threads: thread.join()

    print(f"Verarbeitung und Speicherung abgeschlossen.")

In [2]:
import dask.array as da

In [4]:
raw_darr = da.from_zarr('MRI-Datasets/DCE')
raw_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000), dtype=float32, chunksize=(256, 256, 1, 1), chunktype=numpy.ndarray>

In [5]:
coreg_darr = da.from_zarr('MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr')
coreg_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000), dtype=float32, chunksize=(256, 256, 1, 1), chunktype=numpy.ndarray>

In [6]:
transfo_darr = da.from_zarr('MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr')
transfo_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000, 3), dtype=float32, chunksize=(256, 256, 1, 1, 3), chunktype=numpy.ndarray>

In [7]:
disp_darr = da.from_zarr('MRI_Data/SpatialTransformer_3/predicted_dvfs.zarr')
disp_darr

FileNotFoundError: file://MRI_Data/SpatialTransformer_3/predicted_dvfs.zarr

In [9]:
warped_darr = da.from_zarr('MRI_Data/SpatialTransformer_3/warped_images.zarr')
warped_darr

FileNotFoundError: file://MRI_Data/SpatialTransformer_3/warped_images.zarr

In [10]:
import cupy as cp
import helpers

In [11]:
slice = 27

In [12]:
raw = raw_darr[:,:,slice,:].compute()
raw = cp.transpose(raw, [2,1,0])

In [13]:
coreg = coreg_darr[:,:,slice,:].compute()
coreg = cp.transpose(coreg, [2,1,0])

In [14]:
warped = warped_darr[:,:,slice,:].compute()
warped = cp.transpose(warped,[2,1,0])

NameError: name 'warped_darr' is not defined

In [ ]:
transfo = transfo_darr[:,:,slice,:,0]
transfo = cp.transpose(transfo, [2,1,0])

In [ ]:
disp = disp_darr[:,:,slice,:,0].compute()
disp = cp.transpose(disp, [2,1,0])

In [ ]:
helpers.explore_3D_array_comparison_and_diff(coreg, warped, 'twilight')

interactive(children=(IntSlider(value=500, description='Slice:', max=999), Output()), _dom_classes=('widget-in…

In [ ]:
helpers.explore_3D_array_comparison_and_diff(raw, warped)

interactive(children=(IntSlider(value=500, description='Slice:', max=999), Output()), _dom_classes=('widget-in…

In [15]:
helpers.explore_3D_array_comparison_and_diff(transfo, disp)

NameError: name 'transfo' is not defined

In [16]:
disp_darr.to_zarr('/media/shooty/Data/MRI_Data/SpatialTransformer_3/disp_fields')

NameError: name 'disp_darr' is not defined